# Verifying the implementation of logical Cliffords

Motivation: Logical Cliffords can have non-obvious implementations, especially for $k>1$ codes. We want to be able to validate implementations and catch bugs in our logical gadgets.

Example: check out this paper on CSD codes from N. Berthusen and E. Durso-Sabina (Table 2) -> https://arxiv.org/pdf/2510.18753. Below we will show a section of this table for some logical Clifford operations in the non-CSS $[[4, 2, 2]]$ code.

In [1]:
from IPython.display import Image, display
table_img = Image(url= "images/non_css_4_2_2_logical_cliffords.png", width=300, height=600)

display(table_img)


## What are we verifying?

* This is a simple tool which checks logical Clifford semantics against physical implementation
* Works for $k>1$ codes and also interblock operations
* This approach is fairly general and works for any stabilizer code. Includes non-CSS codes

We will specify the logical semantics with a Guppy function which acts on an array of $k$ qubits. Let's see an example for the $[[7, 1, 3]]$ Steane code. We specify the action of the logical Hadamard on a *logical* qubit as follows...

In [2]:
from typing import no_type_check
from guppylang import guppy
from guppylang.std.builtins import array
from guppylang.std.quantum import qubit, h

@guppy
@no_type_check
def steane_specify_h(qs: array[qubit, 1]) -> None:
    h(qs[0])

We can provide the implementation of the logical operation as a function which acts on an array of $n$ *physical* qubits.

In [3]:
@guppy
@no_type_check
def steane_impl_h(block: array[qubit, 7]) -> None:
    for i in range(len(block)):
        h(block[i])

The verifier then calculates the Clifford tableau of both of them, and checks that they are equivalent. Since the logical specification is in terms of $k$ logical qubits and the implementation is in terms of $n$ logical qubits, we need to expand the tableau of $k$ logical qubits to $n$ physical, using the definition of the stabilizer code. The definition of the stabilizer code tells us how to replace a logical Pauli with its physical Pauli string realisation, as well as which additional Pauli strings we need to introduce to account for the code's stabilizer group.

## Providing the stabilizer code definition

Before we can get to verifying logical Cliffords, we need a way to specify the key properties of a stabilizer code. For this, we use a `StabilizerCode` dataclass which specifies the $[[n, k, d]]$ code parameters as integers as well as the stabilizer generators and logical operators.

```python
@dataclass(frozen=True)
class StabilizerCode:
    num_physical_qubits: int
    num_logical_qubits: int
    distance: int
    generators: pauli.SignTermSet
    x_logicals: pauli.SignTerms
    z_logicals: pauli.SignTerms
```

The stabilizer generators and logical operators are represented using types from [Zixy](https://github.com/Quantinuum/zixy), a library which provides memory efficient Pauli operator implementations in Rust.

A `StabilizerCode` definition can be given by specifying the fields directly. However its usually easier to use the `from_python_strings` utility method.

Let's provide definitions for the Steane code and the CSS $[[4, 2, 2]]$ code.

In [4]:
from guppyft.code_def import StabilizerCode

STEANE_DEF = StabilizerCode.from_python_strings(
    num_physical_qubits=7,
    num_logical_qubits=1,
    distance=3,
    generators=["XXXXIII", "IXXIXXI", "IIXXIXX", "ZZZZIII", "IZZIZZI", "IIZZIZZ"],
    x_logicals=["XXXXXXX"],
    z_logicals=["ZZZZZZZ"],
)



CSS_4Q_DEF = StabilizerCode.from_python_strings(
    num_physical_qubits=4,
    num_logical_qubits=2,
    distance=2,
    generators=["XXXX", "ZZZZ"],
    x_logicals=["XXII", "XIXI"],
    z_logicals=["IZIZ", "IIZZ"],
)



There is some basic `__post_init__` validation to check we are providing reasonable `StabilizerCode` definitions. 

For example, all of the stabilizer generators must commute with one another. If they do not mutually commute, then the code definition is rejected.

In [5]:
INVALID_DEF = StabilizerCode.from_python_strings(
    num_physical_qubits=4,
    num_logical_qubits=2,
    distance=2,
    generators=["XXXX", "XZZZ"], # Stabilizer generators don't commute!
    x_logicals=["XXII", "XIXI"],
    z_logicals=["IZIZ", "IIZZ"],
)

CodeDefinitionError: All of the stabilizer generators must commute!

We can also provide a definition for the $[[5, 1, 3]]$ code. This is a non-CSS code with signed logical operators.

In [8]:
CODE_DEF_5_1_3 = StabilizerCode.from_python_strings(
    num_physical_qubits=5,
    num_logical_qubits=1,
    distance=3,
    generators=["ZZXIX", "XZZXI", "IXZZX", "XIXZZ"],
    x_logicals=["-YIXIY"], # Signed logical operator
    z_logicals=["-XIZIX"], # Signed logical operator
)

## Basic examples for the Steane code

Now we can get on to verifying logical operations. We'll start by verifying some operations functions in the Steane code. This is done with the `check_clifford_semantics` function.

In [9]:
from guppyft.verifier import check_clifford_semantics

First, let's check the logical Hadamard implementation we defined above.

In [10]:
check_clifford_semantics(steane_specify_h, steane_impl_h, STEANE_DEF) # No Error => implementation is valid

We can also verify operators between two code blocks. As an example, let's verify the implementation of the transversal CX in the Steane code.

In [11]:
from guppylang.std.quantum import cx

@guppy
@no_type_check
def steane_specify_cx(first_block: array[qubit, 1], second_block: array[qubit, 1]) -> None:
    cx(first_block[0], second_block[0])


@guppy
@no_type_check
def steane_impl_cx(first_block: array[qubit, 7], second_block: array[qubit, 7]) -> None:
    for i in range(len(first_block)):
        cx(first_block[i], second_block[i])

In [12]:
check_clifford_semantics(steane_specify_cx, steane_impl_cx, STEANE_DEF) # Transversal CX impl is valid.

## Less trivial examples for $k>1$

The examples we have seen so far with the Steane code have just been simple transversal operations. However the implementation of logical Cliffords for $k>1$ codes can be non-obvious. Let's see some examples with the CSS $[[4, 2, 2]]$ code.

In this $[[4, 2, 2]]$ code, we can implement a logical CX gate (intrablock) by simply swapping the physical qubits. 

In [13]:
from guppylang.std.quantum import cx
from guppylang.std.mem import mem_swap

@guppy
@no_type_check
def specify_intra_block_cx(block: array[qubit, 2]) -> None:
    cx(block[0], block[1])


@guppy
@no_type_check
def implement_intra_block_cx(block: array[qubit, 4]) -> None:
    mem_swap(block[3], block[1])

In [14]:
check_clifford_semantics(
    specify_intra_block_cx, implement_intra_block_cx, CSS_4Q_DEF
)

We can also verify the implementation of an $Rx(pi/2)$ gate applied to one of the logical qubits. This operation is implemented with a two-qubit Pauli rotation.

In [15]:
from guppylang.std.quantum import rx
from guppylang.std.angles import pi
from guppylang.std.qsystem.helios import zz_max

@guppy
@no_type_check
def specify_addressable_rx_half_pi(block: array[qubit, 2]) -> None:
    rx(block[1], pi / 2)


@guppy
@no_type_check
def implement_addressable_rx_half_pi(block: array[qubit, 4]) -> None:
    """Non FT"""
    h(block[0])
    h(block[2])
    zz_max(block[0], block[2])
    h(block[0])
    h(block[2])

In [16]:
check_clifford_semantics(
    specify_addressable_rx_half_pi,
    implement_addressable_rx_half_pi,
    CSS_4Q_DEF,
)

The testing framework also allows us to verify state preparation. For example we can check preparation of logical $|0\rangle$ in the $[[4, 2, 2]]$ code as follows.

In [17]:
from guppyft.verifier import check_stabilizer_state_semantics

@guppy
@no_type_check
def specify_zero_state() -> array[qubit, 2]:
    return array(qubit() for _ in range(2))


@guppy
@no_type_check
def implement_non_ft_zero_state() -> array[qubit, 4]:
    """Non fault-tolerant zero state preparation."""
    block = array(qubit() for _ in range(4))

    h(block[0])
    cx(block[0], block[1])
    cx(block[0], block[2])
    cx(block[0], block[3])
    return block



In [18]:
check_stabilizer_state_semantics(specify_zero_state, implement_non_ft_zero_state, CSS_4Q_DEF)

## Explaining the verifier step by step

Let's go back to the logical Hadamard in Steane

In [19]:
from guppyft.verifier import compute_stabilizers_single_block_unitary
from guppyft.verifier.verify import get_expanded_stabilizer_set
from guppyft.code_def import identity_code


# Get the 2k stabilizers for the 2k qubit Choi state encoding the logical operation.
semantic_choi_stabilizers = compute_stabilizers_single_block_unitary(
    code=identity_code(k=1),
    clifford_func=steane_specify_h,
    num_selene_qubits=2*STEANE_DEF.num_logical_qubits,
)

In [20]:
semantic_choi_stabilizers.to_dataframe()

,Component,Coefficient
0,X0 Z1,+1
1,Z0 X1,+1


Expand the $2k$ logical stabilizers to $2k$ stabilizers of size $2n$.
We also add the $2(n-k)$ stabilizer generators of our code.
For each code block there are $n-k$ generators, so $2$ blocks give us $2(n-k)$.
We have $2k + 2(n-k) = 2n$ stabilizers in total.

In [21]:
expanded_semantic_stabilizers = get_expanded_stabilizer_set(
        semantic_choi_stabilizers, STEANE_DEF, num_blocks=2
    )

$$
X_L \mapsto XXXXXXX\, \qquad Z_L \mapsto ZZZZZZZ
$$

In [22]:
expanded_semantic_stabilizers.to_dataframe()

,Component,Coefficient
0,X0 X1 X2 X3 X4 X5 X6 Z7 Z8 Z9 Z10 Z11 Z12 Z13,+1
1,Z0 Z1 Z2 Z3 Z4 Z5 Z6 X7 X8 X9 X10 X11 X12 X13,+1
2,X0 X1 X2 X3,+1
3,X1 X2 X4 X5,+1
4,X2 X3 X5 X6,+1
5,Z0 Z1 Z2 Z3,+1
6,Z1 Z2 Z4 Z5,+1
7,Z2 Z3 Z5 Z6,+1
8,X7 X8 X9 X10,+1
9,X8 X9 X11 X12,+1


In [23]:
expanded_semantic_stabilizers.canonicalize_all()  # Normalize Clifford tableau
expanded_semantic_stabilizers.to_dataframe()

,Component,Coefficient
0,X0 X3 X6 Z11 Z12 Z13,+1
1,X1 X3 X5 Z11 Z12 Z13,+1
2,X2 X3 X5 X6,+1
3,X4 X5 X6 Z11 Z12 Z13,+1
4,Z4 Z5 Z6 X7 X10 X13,+1
5,Z4 Z5 Z6 X8 X10 X12,+1
6,X9 X10 X12 X13,+1
7,Z4 Z5 Z6 X11 X12 X13,+1
8,Z0 Z3 Z4 Z5,+1
9,Z1 Z3 Z4 Z6,+1


In [24]:
# Calculate the 2n stabilizers of the Choi state encoding the physical operation.
implementation_stabilizers = compute_stabilizers_single_block_unitary(
        STEANE_DEF,
        steane_impl_h,
        num_selene_qubits=2 * (STEANE_DEF.num_physical_qubits)
    )

In [25]:
implementation_stabilizers.to_dataframe()

,Component,Coefficient
0,X0 X3 X6 Z11 Z12 Z13,+1
1,Z0 Z3 Z6 X11 X12 X13,+1
2,X1 X3 X5 Z11 Z12 Z13,+1
3,Z1 Z3 Z5 X11 X12 X13,+1
4,X2 X3 X5 X6,+1
5,Z2 Z3 Z5 Z6,+1
6,X4 X5 X6 Z11 Z12 Z13,+1
7,Z4 Z5 Z6 X11 X12 X13,+1
8,X7 X10 X11 X12,+1
9,Z7 Z10 Z11 Z12,+1


In [26]:
implementation_stabilizers.canonicalize_all()  # Normalize Clifford tableau
implementation_stabilizers.to_dataframe()

,Component,Coefficient
0,X0 X3 X6 Z11 Z12 Z13,+1
1,X1 X3 X5 Z11 Z12 Z13,+1
2,X2 X3 X5 X6,+1
3,X4 X5 X6 Z11 Z12 Z13,+1
4,Z4 Z5 Z6 X7 X10 X13,+1
5,Z4 Z5 Z6 X8 X10 X12,+1
6,X9 X10 X12 X13,+1
7,Z4 Z5 Z6 X11 X12 X13,+1
8,Z0 Z3 Z4 Z5,+1
9,Z1 Z3 Z4 Z6,+1


In [27]:
expanded_semantic_stabilizers == implementation_stabilizers

True

## Summary of available features

The following features are available for verification of Clifford operations with `check_clifford_semantics`

* Works for $k>1$ codes
* Works for operations on a single logical block or between two logical blocks
* Works for non-CSS codes (e.g. the $[[5, 1, 3]]$ code)
* Ancilla qubits can be used in the implementation

The features above are also available in `check_stabilizer_state_semantics` which can be used to verify the preparation of logical stabilizer states (e.g. $|0\rangle_L$, $|+\rangle_L$)